To move from our model comparison table to a live list of units on a mechanic's workbench,
we need to bridge the gap between Probability and Policy. 

While our comparison table shows that LightGBM provides a balanced recall of 61.15%, the end user (the Fleet Manager) needs a prioritized, actionable list.

In [4]:
import pickle

# to load the model
with open('../models/pickle/best_model_lightGBM.pkl', 'rb') as f:
     lgbm_model = pickle.load(f)

Here is the three-step workflow to reach that final "Repair Today" list:

step 1. The Decision Engine (Mapping Scores to Tiers)

After our comparison script identifies the best model, we use its probability outputs to categorize every vehicle. Using the thresholds we established, we convert the math into a "Traffic Light" system:

In [5]:
with open('../models/pickle/X_test_temporal.pkl', 'rb') as f:
     X_latest = pickle.load(f)

In [ ]:
X_latest # import daily data 

In [ ]:
import numpy as np
import pandas as pd

# Assuming 'lgbm_model' is your best estimator and X_latest is the daily sensor data
# 1. Generate probabilities for the entire fleet
probs = lgbm_model.predict_proba(X_latest)[:, 1]

# 2. Create the Decision Table
# 'fleet_unit_ids' would be your VINs or Vehicle Numbers
fleet_status = pd.DataFrame({
    'Unit_ID': asset_id, # The unique vehicle identifier
    'Probability': probs
})

# 3. Apply the Business Policy
conditions = [
    (fleet_status['Probability'] > 0.55),
    (fleet_status['Probability'] >= 0.35) & (fleet_status['Probability'] <= 0.55),
    (fleet_status['Failure_Probability'] < 0.35)
]
statuses = ['CRITICAL', 'WATCHLIST','HEALTHY']
actions = ['URGENT: Repair Today', 'Monitor: Inspect within 48h', 'No immediate action required']

fleet_status['Status'] = np.select(conditions, statuses, default='HEALTHY')
fleet_status['Instruction'] = np.select(conditions, actions, default='Standard Operations')

# Filter for the end-user
daily_repair_list = fleet_status[fleet_status['Status'] != 'HEALTHY'].sort_values('Probability', ascending=False)

2. Adding "The Why" (SHAP Integration)
A mechanic won't trust a "Black Box" list. To make the list useful, we attach the top features identified in our Feature Importance analysis (like days_since_service, odometer, and oil_press_std_7d).

Final List Item Example:

Unit ID: #TRK-882

Status: CRITICAL

Reason: High Vibration (Top Feature) & Oil Pressure Volatility.

3. The End-User Interface (The Dashboard)
The results from the code above are pushed to a dashboard (Power BI or a custom web app). This is what the Fleet Manager sees every morning:

| Asset ID | Risk Level | Primary Trigger | Action Required |
| :-- | :-- | :-- | :-- |
| #TRK-882 | "<span style=""color:red"">CRITICAL</span>" | Vibration Variance | Pull from route immediately |
| #TRK-104 | "<span style=""color:red"">CRITICAL</span>" | Days Since Service | Inspect Oil/Filters today |
| #TRK-442 | "<span style=""color:orange"">WATCHLIST</span>" | High Odometer | Schedule for weekend check |


**From Code to the Mechanic's Workbench**

In a real business, that repair_today_list is the data source for a live dashboard. Here is how the end-user interacts with our work:

Step A: The Morning Briefing

The Fleet Manager opens their dashboard (Power BI or a custom app). The model has automatically sorted thousands of vehicles into a top 10 list.

Step B: The "Diagnostic Hook"

Because we performed SHAP analysis, the dashboard doesn't just say "Fix Unit #882." It shows a tooltip:

"High Risk Triggered by: Oil Pressure Volatility & Vibration Standard Deviation."

This gives the mechanic a starting point, saving them hours of manual troubleshooting.

Step C: The Feedback Loop (The "Real-World" End)

The mechanic repairs the vehicle and clicks "Closed" in the system.

If they found a worn bearing: True Positive (The model saved a breakdown).

If they found nothing: False Positive (This data is sent back to you to retrain and improve the model next month).


**Executive Summary (Technical Summary)**

By operationalizing the LightGBM model results, we transition from a static evaluation to a dynamic Prioritized Maintenance Queue. 

Selection: Chose LightGBM based on its 61% recall and efficient handling of class imbalance.

Probability Thresholding: Converted raw model output into a 3-tier risk system (Critical, Watchlist, Healthy).

Actionability: Integrated feature importance into the dashboard to provide "Explainable AI" for mechanics.

Operational Result: the maintenance department can focus labor on the top 40% of high-risk units, effectively capturing 61% of potential failures before they lead to roadside downtime